# 02 - Generate OSS Model Responses

This is the main OCN generation experiment. It runs matched base/post-trained pairs from Gemma 4 E2B and Qwen 3.5 2B over the full prompt bank, autosaves resumable chunks to Google Drive, logs progress to W&B, and publishes to a dedicated Hugging Face dataset.

Required runtime: an A100 GPU. The notebook stops before generation if Colab assigned a different accelerator. Before running, accept access to both `google/gemma-4-E2B` and `google/gemma-4-E2B-it` on Hugging Face.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import gc, json
from importlib.metadata import version
from pathlib import Path
from packaging.version import Version
import pandas as pd
import torch
import wandb
from datasets import load_dataset

from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, publish_dataframe_to_hf, save_dataframe, utc_timestamp
from ocn.generation import (
    DecodingSpec,
    ModelSpec,
    generation_rows,
    load_text_generation_model,
)
from ocn.prompt_factory import slugify

if Version(version("transformers")) < Version("5.14.1"):
    raise RuntimeError(
        "Gemma 4 requires transformers>=5.14.1. Start a fresh Colab runtime, "
        "run the updated notebook 00, then return to notebook 02."
    )

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime > Change runtime type > A100 GPU.")
GPU_NAME = torch.cuda.get_device_name(0)
if "A100" not in GPU_NAME.upper():
    raise RuntimeError(
        f"This main run requires an A100, but Colab assigned {GPU_NAME}. "
        "Reconnect with Runtime > Change runtime type > A100 GPU."
    )
print("Verified accelerator:", GPU_NAME)

paths = make_colab_paths()
config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
login_huggingface("HF_WRITE_ACCESS")

In [ ]:
prompts = load_dataset(config["hf_prompt_repo"], split="train").to_pandas()

EXPERIMENT_ID = "main_gemma4_qwen35"
GENERATION_RUN_ID = utc_timestamp()
MAIN_GENERATION_REPO = config.get(
    "hf_main_generation_repo",
    f"{config['hf_owner']}/ocn-empty-negations-generations-main-gemma4-qwen35",
)

MODEL_SPECS = [
    ModelSpec("google/gemma-4-E2B", "gemma4", "base", False, "multimodal_lm"),
    ModelSpec("google/gemma-4-E2B-it", "gemma4", "instruct", True, "multimodal_lm"),
    ModelSpec("Qwen/Qwen3.5-2B-Base", "qwen3.5", "base", False, "multimodal_lm"),
    ModelSpec("Qwen/Qwen3.5-2B", "qwen3.5", "instruct", True, "multimodal_lm"),
]

DECODINGS = [
    DecodingSpec("greedy", temperature=0.0, top_p=1.0, max_new_tokens=160),
    DecodingSpec("normal_temp", temperature=0.7, top_p=0.95, max_new_tokens=180),
]

SEEDS = config["default_seeds"]
PROMPT_BATCH_SIZE = 24
# All four models fit sequentially in BF16 on an A100. Do not enable
# quantization for the main study because it changes the model itself.
QUANTIZE_4BIT = False
EXPECTED_ROWS = len(prompts) * len(MODEL_SPECS) * len(DECODINGS) * len(SEEDS)

experiment_config = {
    **config,
    "experiment_id": EXPERIMENT_ID,
    "generation_run_id": GENERATION_RUN_ID,
    "run_mode": "main",
    "gpu_name": GPU_NAME,
    "precision": "bfloat16",
    "quantize_4bit": QUANTIZE_4BIT,
    "main_generation_repo": MAIN_GENERATION_REPO,
    "models": [spec.model_id for spec in MODEL_SPECS],
    "prompt_count": len(prompts),
    "expected_rows": EXPECTED_ROWS,
    "prompt_batch_size": PROMPT_BATCH_SIZE,
}
run = login_wandb(
    project="ocn-empty-negations",
    name=f"generate-{EXPERIMENT_ID}-{GENERATION_RUN_ID}",
    config=experiment_config,
)
print("Prompts:", len(prompts), "Models:", len(MODEL_SPECS), "Decodings:", len(DECODINGS), "Seeds:", SEEDS)
print("Expected rows:", EXPECTED_ROWS)
print("Publishing to:", MAIN_GENERATION_REPO)

In [ ]:
generation_dir = Path(config["drive_data_root"]) / "generation_runs" / EXPERIMENT_ID
generation_dir.mkdir(parents=True, exist_ok=True)
expected_part_rows = len(prompts) * len(SEEDS)
expected_part_keys = {
    (prompt_id, seed)
    for prompt_id in prompts["prompt_id"]
    for seed in SEEDS
}

def part_path_for(model_spec, decoding):
    model_slug = slugify(model_spec.model_id)
    return generation_dir / f"{model_slug}_{decoding.name}.csv"

def load_existing_part(path):
    if not path.exists():
        return pd.DataFrame()
    part = pd.read_csv(path)
    required = {"prompt_id", "seed", "model_id", "decoding", "response"}
    if not required.issubset(part.columns):
        print(f"Ignoring incompatible checkpoint: {path.name}")
        return pd.DataFrame()
    part = part[part["prompt_id"].isin(prompts["prompt_id"])].copy()
    return part.drop_duplicates(["prompt_id", "seed"], keep="last")

def load_complete_part(path):
    part = load_existing_part(path)
    keys = set(zip(part.get("prompt_id", []), part.get("seed", [])))
    if keys != expected_part_keys:
        return None
    return part

def combine_complete_parts():
    frames = []
    for spec in MODEL_SPECS:
        for decoding_spec in DECODINGS:
            complete = load_complete_part(part_path_for(spec, decoding_spec))
            if complete is not None:
                frames.append(complete)
    if not frames:
        raise RuntimeError("No completed generation parts were found.")
    return pd.concat(frames, ignore_index=True).sort_values(
        ["model_id", "decoding", "prompt_id", "seed"]
    ).reset_index(drop=True)

for model_spec in MODEL_SPECS:
    missing_decodings = [
        decoding for decoding in DECODINGS
        if load_complete_part(part_path_for(model_spec, decoding)) is None
    ]
    if not missing_decodings:
        print(f"Skipping completed model: {model_spec.model_id}")
        continue

    print(f"\nLoading {model_spec.model_id}")
    try:
        tokenizer, model = load_text_generation_model(
            model_spec.model_id,
            quantize_4bit=QUANTIZE_4BIT,
            loader_type=model_spec.loader_type,
        )
    except Exception as exc:
        access_hint = (
            " Accept the Gemma model terms on Hugging Face and ensure "
            "HF_WRITE_ACCESS can read gated models."
            if model_spec.model_id.startswith("google/gemma") else ""
        )
        raise RuntimeError(f"Could not load {model_spec.model_id}.{access_hint}") from exc

    model_revision = getattr(model.config, "_commit_hash", None)
    for decoding in missing_decodings:
        print(f"Generating: {model_spec.model_id} / {decoding.name}")
        part_path = part_path_for(model_spec, decoding)
        part = load_existing_part(part_path)
        completed_prompt_ids = {
            prompt_id
            for prompt_id, group in part.groupby("prompt_id")
            if set(group["seed"]) == set(SEEDS)
        } if not part.empty else set()
        remaining = prompts[~prompts["prompt_id"].isin(completed_prompt_ids)]

        for start in range(0, len(remaining), PROMPT_BATCH_SIZE):
            prompt_batch = remaining.iloc[start : start + PROMPT_BATCH_SIZE]
            rows = generation_rows(
                prompts=prompt_batch,
                model_spec=model_spec,
                decoding=decoding,
                tokenizer=tokenizer,
                model=model,
                seeds=SEEDS,
            )
            new_rows = pd.DataFrame(rows)
            new_rows["experiment_id"] = EXPERIMENT_ID
            new_rows["generation_run_id"] = GENERATION_RUN_ID
            new_rows["gpu_name"] = GPU_NAME
            new_rows["precision"] = "bfloat16"
            new_rows["model_revision"] = model_revision
            part = pd.concat([part, new_rows], ignore_index=True).drop_duplicates(
                ["prompt_id", "seed"], keep="last"
            )
            save_dataframe(part, part_path)
            wandb.log({
                "checkpoint_rows": len(part),
                "checkpoint_fraction": len(part) / expected_part_rows,
                "model_id": model_spec.model_id,
                "decoding": decoding.name,
            })
            print(f"Checkpointed {len(part)}/{expected_part_rows}: {part_path.name}")

        part = load_complete_part(part_path)
        if part is None:
            raise RuntimeError(f"Generation checkpoint is incomplete: {part_path}")

        combined = combine_complete_parts()
        combined_path = save_dataframe(
            combined,
            Path(config["drive_data_root"]) / "ocn_generations_main_gemma4_qwen35.csv",
        )
        repo_url = publish_dataframe_to_hf(
            combined,
            repo_id=MAIN_GENERATION_REPO,
            split="train",
            private=config["hf_private"],
            card_path=REPO_ROOT / "dataset_cards/ocn_generations.md",
            commit_message=f"Update {EXPERIMENT_ID} generations {GENERATION_RUN_ID}",
        )
        wandb.log({
            "generated_rows": len(combined),
            "expected_rows": EXPECTED_ROWS,
            "completion_fraction": len(combined) / EXPECTED_ROWS,
            "last_part_rows": len(part),
            "model_id": model_spec.model_id,
            "decoding": decoding.name,
        })
        print("Autosaved:", combined_path)
        print("Autopublished:", repo_url)

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

run.finish()